<a href="https://colab.research.google.com/github/UPEI-CS/CS1910-Students/blob/main/case_study/Wordle/CS1910_Wordle_Workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS1910 Computer Science 1: Case Study**
# **Wordle**

## Save your own copy first

Only the **computer recorder** needs to do this, but everyone may want a copy afterwards. **Do this before you start**, because anything you type into the original is not saved. Choose:

***File->Save a copy in Drive***

or

***Copy to Drive*** if it appears in the above menu.

Google sign-in is required. You will then be able to open the copy from your Google Drive account later (Colab keeps it in the *Colab Notebooks* folder). At the end, share your copy with the rest of your group (**Share** at the top right).

If you can't use Google Drive, choose ***File->Download .ipynb*** and keep the file by email (signing in to a cloud email service) or on a USB key. To open it in another Colab session later, you would have to move it into your Google Drive account.

## How to use this notebook

This is the coding half of the Wordle case study. You work in a **group of 3–5**: the **paper recorder** fills in the group worksheet (`CS1910_Wordle-Partner_Handout.pdf`), the **computer recorder** types in this notebook, and the **speaker** reports to the class. Plan on paper first, then code: each task below says which worksheet stage to do before it.

1. **Run the setup cell** (▶️ below) first. It gives you a `check` helper that tells you ✅ or ❌.
2. **Run the word-list cell.** It picks a secret word for the game.
3. **Work through the tasks in order.** For each one, answer the *Understand* questions, write your plan in the ✏️ cell (double-click to edit), then code and run the checks.

Make sure Colab's AI assistance is **off**: this is code your group writes.

## Wordle in one minute

The secret is a 5-letter word and you get 6 guesses. After each guess, each letter is coloured:

- **Green (G):** the letter is in the secret in this exact position.
- **Yellow (Y):** the letter is in the secret, but somewhere else.
- **Black (B):** the letter is not in the secret, or every copy of it has already been used.

Score the **greens first**, then the yellows from **left to right**. Each letter of the secret can be used by only **one** letter of the guess. Guesses that are not 5-letter words in the list are ignored and do not use up a try.

For example, with secret `silly` the guess `isles` scores `YYGBB`: the `l` in the middle is green, the `i` and the first `s` are yellow, and the second `s` is black because the secret's only `s` is already used.

Wordle is a version of the much older code-breaking games [Mastermind](https://en.wikipedia.org/wiki/Mastermind_(board_game)) and [Bulls and Cows](https://en.wikipedia.org/wiki/Bulls_and_cows). If you have never played, try a round with your group before you start.

In [ ]:
#@title ▶️ Run this cell first
#@markdown This cell sets up two helpers, `check` and `check_case`, that tell you
#@markdown whether your answer is right. You do not need to understand how they work.
import math

_cases_passed = {}


def _same(got, expected):
    if isinstance(expected, float) and isinstance(got, (int, float)) and not isinstance(got, bool):
        return math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-12)
    return type(got) == type(expected) and got == expected


def check(got, expected, label="Answer"):
    if _same(got, expected):
        print(f"✅ {label}: {got!r}")
    elif got == expected:
        print(f"❌ {label}: the value looks right, but it is a {type(got).__name__}"
              f" and should be a {type(expected).__name__}")
    else:
        print(f"❌ {label}: got {got!r}, expected {expected!r}")


def check_case(name, case, got, expected_by_case):
    if case not in expected_by_case:
        print(f"ℹ️ {case!r} is not one of the test cases for {name}. Try one of: "
              + ", ".join(repr(c) for c in expected_by_case))
        return
    passed = _cases_passed.setdefault(name, set())
    check(got, expected_by_case[case], f"{name} with {case!r}")
    if _same(got, expected_by_case[case]):
        passed.add(case)
    still = [repr(c) for c in expected_by_case if c not in passed]
    if still:
        print(f"   {len(passed)} of {len(expected_by_case)} test cases passed. "
              f"Change the input and run again. Still to try: {', '.join(still)}")
    else:
        print(f"   🎉 All {len(expected_by_case)} test cases passed!")


print("✅ Practice checker ready.")

## Colour in the output

The `rich` module lets `print` show colour. `rich.print` reads tags such as `[green]...[/green]` in the string. Run the cell to see a scored guess.

In [ ]:
import rich

print("... s i l l y ...")
print("... i s l e s ...")
rich.print("... [gold3]■[/gold3] [gold3]■[/gold3] [green]■[/green] [black]■[/black] [black]■[/black] ...")

## The word list

Run this cell to load the words and choose a secret.

In [ ]:
# The word list: every secret and every guess must be one of these words.
# (Generative AI produced this list of common 5-letter words.)

words = [
    "about", "above", "abuse", "actor", "acute", "admit", "adopt", "adult", "after", "again",
    "agent", "agree", "ahead", "alarm", "album", "alert", "alike", "alive", "allow", "alone",
    "along", "alter", "among", "anger", "angle", "angry", "apart", "apple", "apply", "arena",
    "argue", "arise", "armed", "array", "aside", "asset", "audio", "audit", "avoid", "award",
    "aware", "badly", "baker", "bases", "basic", "basis", "beach", "began", "begin", "begun",
    "being", "below", "bench", "billy", "birth", "black", "blame", "blind", "block", "blood",
    "board", "boost", "booth", "bound", "brain", "brand", "bread", "break", "breed", "brief",
    "bring", "broad", "broke", "brown", "build", "built", "buyer", "cable", "calif", "carry",
    "catch", "cause", "chain", "chair", "chart", "chase", "cheap", "check", "chest", "chief",
    "child", "china", "chose", "civil", "claim", "class", "clean", "clear", "click", "clock",
    "close", "coach", "coast", "could", "count", "court", "cover", "craft", "crash", "crazy",
    "cream", "crime", "cross", "crowd", "crown", "curve", "cycle", "daily", "dance", "dated",
    "dealt", "death", "debut", "delay", "depth", "doing", "doubt", "dozen", "draft", "drama",
    "drawn", "dream", "dress", "drill", "drink", "drive", "drove", "dying", "eager", "early",
    "earth", "eight", "elite", "empty", "enemy", "enjoy", "enter", "entry", "equal", "error",
    "event", "every", "exact", "exist", "extra", "faith", "false", "fault", "fiber", "field",
    "fifth", "fifty", "fight", "final", "first", "fixed", "flash", "fleet", "floor", "fluid",
    "focus", "force", "forth", "forty", "forum", "found", "frame", "frank", "fraud", "fresh",
    "front", "fruit", "fully", "funny", "giant", "given", "glass", "globe", "going", "grace",
    "grade", "grand", "grant", "grass", "great", "green", "gross", "group", "grown", "guard",
    "guess", "guest", "guide", "happy", "harsh", "heart", "heavy", "hence", "henry", "horse",
    "hotel", "house", "human", "ideal", "image", "index", "inner", "input", "issue", "joint",
    "judge", "known", "label", "large", "laser", "later", "laugh", "layer", "learn", "lease",
    "least", "leave", "legal", "level", "light", "limit", "links", "lives", "local", "logic",
    "loose", "lower", "lucky", "lunch", "lying", "magic", "major", "maker", "march", "match",
    "maybe", "mayor", "meant", "media", "metal", "might", "minor", "minus", "mixed", "model",
    "money", "month", "moral", "motor", "mount", "mouse", "mouth", "movie", "music", "needs",
    "never", "newly", "night", "noise", "north", "noted", "novel", "nurse", "occur", "ocean",
    "offer", "often", "order", "other", "ought", "paint", "panel", "paper", "party", "peace",
    "phase", "phone", "photo", "piece", "pilot", "pitch", "place", "plain", "plane", "plant",
    "plate", "point", "pound", "power", "press", "price", "pride", "prime", "print", "prior",
    "prize", "proof", "proud", "prove", "queen", "quick", "quiet", "quite", "radio", "raise",
    "range", "rapid", "ratio", "reach", "ready", "refer", "relax", "reply", "right", "rival",
    "river", "robin", "robot", "rocky", "roman", "rough", "round", "route", "royal", "rural",
    "scale", "scene", "scope", "score", "sense", "serve", "seven", "shall", "shape", "share",
    "sharp", "sheet", "shelf", "shell", "shift", "shirt", "shock", "shoot", "short", "shown",
    "sight", "since", "sixth", "sixty", "skill", "sleep", "slide", "small", "smart", "smile",
    "smith", "smoke", "solid", "solve", "sorry", "sound", "south", "space", "spare", "speak",
    "speed", "spend", "spent", "split", "spoke", "sport", "staff", "stage", "stake", "stand",
    "start", "state", "steam", "steel", "stick", "still", "stock", "stone", "stood", "store",
    "storm", "story", "strip", "stuck", "study", "stuff", "style", "sugar", "suite", "super",
    "sweet", "table", "taken", "taste", "taxes", "teach", "teeth", "terry", "texas", "thank",
    "theft", "their", "theme", "there", "these", "thick", "thing", "think", "third", "those",
    "three", "threw", "throw", "tight", "times", "tired", "title", "today", "topic", "total",
    "touch", "tough", "tower", "track", "trade", "train", "treat", "trend", "trial", "tried",
    "tries", "truck", "truly", "trust", "truth", "twice", "under", "union", "unity", "until",
    "upper", "upset", "urban", "usage", "usual", "valid", "value", "video", "virus", "visit",
    "vital", "voice", "waste", "watch", "water", "wheel", "where", "which", "while", "white",
    "whole", "whose", "woman", "women", "world", "worry", "worse", "worst", "worth", "would",
    "wound", "write", "wrong", "wrote", "yield", "young", "youth"
]

import random

# random.seed(55)   # uncomment to get the same "random" secret every time while testing
secret = random.choice(words)
print(f"{len(words)} words loaded. A secret word has been chosen.")

## Task 1: Score by hand (worksheet Stages 0–1)

Do the first two stages of the worksheet on paper: fill in your roles, score the six wordles, and find the impossible score. You will use your answers to check your code in Task 4. Then come back here.

## Changing one letter of a string

We will keep two strings while scoring:

- `score` starts as `"BBBBB"`; each `B` is replaced by `G` or `Y` as letters are scored.
- `secret_copy` starts as a copy of the secret; each letter that gets **used** is replaced by `"!"`, so it cannot be used twice.

Strings are **immutable**, so we cannot write `score[2] = "G"`. Instead we build a new string from three pieces:

```python
score = score[:2] + "G" + score[3:]            # put "G" at index 2
secret_copy = secret_copy[:2] + "!" + secret_copy[3:]
```

In general, to put `ch` at index `i`: `s = s[:i] + ch + s[i + 1:]`.

## Task 2: Score the greens (worksheet Stage 3)

Write `score_greens(secret, guess)`. It returns **two** strings, `score` and `secret_copy`, after the greens have been scored: a `"G"` in `score` wherever the guess has the right letter in the right place, and a `"!"` at the same index in `secret_copy`. Do not change `secret` or `guess`.

A function can return two values: `return score, secret_copy`. The call then gives back a **tuple**, which you can unpack: `score, secret_copy = score_greens("occur", "rocky")`.

**Understand:**
- For secret `occur` and guess `rocky`, what should the two strings be? (Your worksheet has the answer.)
- What is `score[:i]` when `i` is 0? What is `score[i + 1:]` when `i` is 4?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def score_greens(secret, guess):
    """Return (score, secret_copy) after scoring the greens."""
    score = "BBBBB"
    secret_copy = secret
    # Your code here

    return score, secret_copy

# ---- Checks: run the cell to test your code (do not change below) ----
check(score_greens("occur", "rocky"), ("BBGBB", "oc!ur"), 'score_greens("occur", "rocky")')
check(score_greens("guess", "swiss"), ("BBBGG", "gue!!"), 'score_greens("guess", "swiss")')
check(score_greens("apple", "apple"), ("GGGGG", "!!!!!"), 'score_greens("apple", "apple")')
check(score_greens("abcde", "fghij"), ("BBBBB", "abcde"), 'score_greens("abcde", "fghij")')

## Task 3: Score the yellows (worksheet Stage 4)

Write `score_yellows(guess, score, secret_copy)`. It takes the two strings from `score_greens` and returns them again after the yellows are scored. For each letter of the guess that is **not** already green, look through `secret_copy` for the same letter. At the **first** match, put `"Y"` in `score` at the guess's index, put `"!"` in `secret_copy` at the match's index, and stop looking.

**Understand:**
- Why must you search `secret_copy`, not `secret`?
- Which index goes into `score`, and which into `secret_copy`?
- Why must you stop at the first match?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def score_yellows(guess, score, secret_copy):
    """Return (score, secret_copy) after scoring the yellows."""
    # Your code here

    return score, secret_copy

# ---- Checks: run the cell to test your code (do not change below) ----
check(score_yellows("rocky", "BBGBB", "oc!ur"), ("YYGBB", "!c!u!"), "occur / rocky")
check(score_yellows("isles", "BBGBB", "si!ly"), ("YYGBB", "!!!ly"), "silly / isles")
check(score_yellows("swiss", "BBBGG", "gue!!"), ("BBBGG", "gue!!"), "guess / swiss")
check(score_yellows("board", "BBBBB", "about"), ("YYYBB", "!!!ut"), "about / board")

## Task 4: Put the scoring together

Write `score_guess(secret, guess)`, which returns only the final score string. It should call your two functions; it needs only a few lines.

The checks use the six wordles from the worksheet, so if one fails, compare the code's answer with your group's hand-worked answer: which one is wrong?

In [ ]:
def score_guess(secret, guess):
    """Return the Wordle score of guess, e.g. "YYGBB"."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(score_guess("oopsy", "opens"), "GYBBY", "worksheet #1")
check(score_guess("error", "rotor"), "YBBGG", "worksheet #2")
check(score_guess("start", "array"), "YYBBB", "worksheet #3")
check(score_guess("ideal", "leads"), "YYYYB", "worksheet #4")
check(score_guess("stats", "twist"), "YBBYY", "worksheet #5")
check(score_guess("guess", "swiss"), "BBBGG", "worksheet #6")
check(score_guess("lever", "eerie"), "YGYBB", "three e's in the guess")

Now try your group's own "hard to score" pair from the worksheet (question 3). Does the code agree with your hand score?

In [ ]:
print(score_guess("abbey", "kebab"))   # change these to your group's pair

## Showing the score in colour

This helper prints a guess with its coloured score. You do not need to change it.

In [ ]:
import rich

def show_score(guess, score):
    """Print the letters of guess, then the score as coloured squares."""
    colours = {"G": "green", "Y": "gold3", "B": "black"}
    squares = ""
    for ch in score:
        squares += f"[{colours[ch]}]■[/{colours[ch]}] "
    print(" ".join(guess), end="   ")
    rich.print(squares)

show_score("isles", score_guess("silly", "isles"))

## Task 5: The game loop (worksheet Stage 2)

Turn your group's pseudocode from Stage 2 into `play_wordle(secret)`. It should:

- allow up to 6 guesses, and stop early when the guess is right;
- ask again, **without** using up a try, when the guess is not a 5-letter word from `words`;
- show each scored guess with `show_score`;
- congratulate the player, or reveal the secret if they run out of guesses.

Then play a few games. There are no automatic checks for this task: test it by playing, including a guess that is too short and a guess that is not a word.

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def play_wordle(secret):
    """Play one game of Wordle against the given secret word."""
    max_guesses = 6
    # Your code here


play_wordle(random.choice(words))

## Share your result without spoiling it

Players share their game as coloured squares only, never the letters. Change the scores below to match a game you played, and run the cell.

In [ ]:
import rich

my_game = ["BYBBB", "BBGYB", "GGGBG", "GGGGG"]   # change to your own scores
colours = {"G": "green", "Y": "gold3", "B": "black"}
for score in my_game:
    rich.print(" ".join(f"[{colours[ch]}]■[/{colours[ch]}]" for ch in score))

## Before you leave

Tick off each statement your group can honestly agree with. For any you can't, go back to the matching task or worksheet stage, and bring your questions to the next class.

- [ ] I can score a Wordle guess by hand, including guesses with repeated letters.
- [ ] I can explain why the greens must be scored before the yellows.
- [ ] I can build a new string to "change" one character of an immutable string.
- [ ] I can return two values from a function and unpack them.
- [ ] I can use `break` to stop an inner loop at the first match.

**Share your copy** of this notebook with the rest of your group before you go.

### Related (not required) reading

- Donald Knuth, *The Computer as Master Mind*: https://www.cs.uni.edu/~wallingf/teaching/cs3530/resources/knuth-mastermind.pdf
- Some instructors suspect that Mastermind ability predicts programming aptitude: *MasterMind(c): A Predictor of Computer Programming Aptitude*, https://dl.acm.org/doi/pdf/10.1145/1138403.1138436